In [1]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

In [9]:
iris = load_iris(as_frame=True)
df = iris.frame

# Feature, Target 분리
X = df.drop(columns="target")
y = df["target"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [12]:
print(X.shape, y.shape)
print(X_train.shape, X_test.shape)

(150, 4) (150,)
(120, 4) (30, 4)


In [16]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

print(X_train_tensor.size())
print(X_test_tensor.size())
print(y_train_tensor.size())
print(y_test_tensor.size())

torch.Size([120, 4])
torch.Size([30, 4])
torch.Size([120])
torch.Size([30])


In [17]:
model = nn.Sequential(
    nn.Linear(X_train_tensor.size(-1), 16),
    nn.ReLU(),
    
    nn.Linear(16, 8),
    nn.ReLU(),
    
    nn.Linear(8, 3),
)

model

Sequential(
  (0): Linear(in_features=4, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=3, bias=True)
)

In [18]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

CrossEntropyLoss(): 모델 출력값과 실제 클래스 번호 비교해 다중 클래스 분류 손실 계산. Softmax() 내장되어 있음

In [19]:
for epoch in range(1000):
    model.train()

    optimizer.zero_grad()
    output = model(X_train_tensor)
    loss = criterion(output, y_train_tensor)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1} - Loss: {loss.item():.4f}')

Epoch 100 - Loss: 0.0335
Epoch 200 - Loss: 0.0234
Epoch 300 - Loss: 0.0215
Epoch 400 - Loss: 0.0210
Epoch 500 - Loss: 0.0207
Epoch 600 - Loss: 0.0205
Epoch 700 - Loss: 0.0204
Epoch 800 - Loss: 0.0204
Epoch 900 - Loss: 0.0201
Epoch 1000 - Loss: 0.0200


In [20]:
model.eval()

with torch.no_grad():
    output = model(X_test_tensor)

print(output[:5])
print(output.shape)

tensor([[  8.5027,  -5.6330,  -9.3730],
        [ -5.9130,   0.3118,   0.5490],
        [ -1.6234,   7.7724, -13.7032],
        [ -3.1600,   9.9665, -15.2883],
        [  8.7329,  -6.3049,  -8.7926]])
torch.Size([30, 3])


In [26]:
with torch.no_grad():
    output = model(X_test_tensor)
    prediction = output.argmax(dim=1)

print(output[:10])
print(prediction[:10])

tensor([[  8.5027,  -5.6330,  -9.3730],
        [ -5.9130,   0.3118,   0.5490],
        [ -1.6234,   7.7724, -13.7032],
        [ -3.1600,   9.9665, -15.2883],
        [  8.7329,  -6.3049,  -8.7926],
        [ -4.1778,   6.7139,  -7.0844],
        [  9.3182,  -7.1481,  -8.6641],
        [  7.2303,  -3.6057,  -9.8160],
        [ -9.8421, -11.8248,  13.2131],
        [ -3.9410,   7.5443,  -8.3204]])
tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1])


In [38]:
with torch.no_grad():
    output = model(X_test_tensor)
    probability = torch.softmax(output, dim=1)
    prediction = output.argmax(dim=1)

print(output[:10])
for r in probability[:10]:
    print([f'{p.item():.6f}' for p in r])
print(prediction[:10])

tensor([[  8.5027,  -5.6330,  -9.3730],
        [ -5.9130,   0.3118,   0.5490],
        [ -1.6234,   7.7724, -13.7032],
        [ -3.1600,   9.9665, -15.2883],
        [  8.7329,  -6.3049,  -8.7926],
        [ -4.1778,   6.7139,  -7.0844],
        [  9.3182,  -7.1481,  -8.6641],
        [  7.2303,  -3.6057,  -9.8160],
        [ -9.8421, -11.8248,  13.2131],
        [ -3.9410,   7.5443,  -8.3204]])
['0.999999', '0.000001', '0.000000']
['0.000872', '0.440590', '0.558538']
['0.000083', '0.999917', '0.000000']
['0.000002', '0.999998', '0.000000']
['1.000000', '0.000000', '0.000000']
['0.000019', '0.999980', '0.000001']
['1.000000', '0.000000', '0.000000']
['0.999980', '0.000020', '0.000000']
['0.000000', '0.000000', '1.000000']
['0.000010', '0.999990', '0.000000']
tensor([0, 2, 1, 1, 0, 1, 0, 0, 2, 1])


In [41]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = prediction.numpy()
y_true = y_test_tensor.numpy()

print(f'Acc: {accuracy_score(y_true, y_pred)}')
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=iris.target_names))


Acc: 0.9333333333333333
[[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.90      0.90      0.90        10
   virginica       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30

